# 1-stage vs 2-stage Validation End-to-End 평가

corrected bbox로 재학습한 두 파이프라인을 같은 validation 이미지에서 비교합니다.

## 파이프라인

- **1-stage**: YOLO 9-class → bbox + 재질/오염도 결합 클래스
- **2-stage**: YOLO 3-material → 예측 bbox를 padding 0.05로 crop → ResNet18 dirty 3-class

## 핵심 평가 원칙

- GT와 예측 bbox가 IoU 0.5 이상일 때만 위치 탐지 성공으로 인정
- 위치를 못 찾은 이미지를 정확도 계산에서 제외하지 않음
- 최종 9-class, 재질, 오염도를 분리해 분석
- confidence threshold를 여러 값에서 비교
- 별도 test split이 없으므로 결과는 **validation 성능**으로 표현


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 설정 및 파일 검사

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw, ImageFont, ImageFile
from tqdm.auto import tqdm
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torchvision import models, transforms
from ultralytics import YOLO

ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

VAL_IMAGE_DIR = Path('/content/drive/MyDrive/TeamProject/test_dataset/1-stage/images/val')
VAL_LABEL_DIR = Path('/content/drive/MyDrive/TeamProject/test_dataset/1-stage/labels/val')

YOLO_1STAGE_PATH = Path(
    '/content/drive/MyDrive/TeamProject/test_dataset/runs/yolov8n_1stage_9class_bboxfixed/weights/best.pt'
)
YOLO_2STAGE_PATH = Path(
    '/content/drive/MyDrive/TeamProject/test_dataset/runs/yolov8n_2stage_material3_bboxfixed/weights/best.pt'
)
CLASSIFIER_PATH = Path(
    '/content/drive/MyDrive/TeamProject/test_dataset/runs/resnet18_dirty3_bboxfixed_pad005/'
    'best_resnet18_dirty3_state_dict.pt'
)

OUTPUT_DIR = Path('/content/drive/MyDrive/TeamProject/test_dataset/runs/end_to_end_bboxfixed_validation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_NAMES = [
    'can_clean', 'can_outer', 'can_inner',
    'pet_clean', 'pet_outer', 'pet_inner',
    'plastic_clean', 'plastic_outer', 'plastic_inner',
]
MATERIAL_NAMES = ['can', 'pet', 'plastic']
DIRT_NAMES = ['clean', 'outer', 'inner']

IOU_THRESHOLD = 0.50
RAW_PREDICTION_CONF = 0.01
NMS_IOU = 0.70
MAX_DETECTIONS = 20
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
COMMON_REPORT_THRESHOLD = 0.25
TWO_STAGE_CROP_PADDING = 0.05
REUSE_PREDICTION_CACHE = True

required_paths = [
    VAL_IMAGE_DIR,
    VAL_LABEL_DIR,
    YOLO_1STAGE_PATH,
    YOLO_2STAGE_PATH,
    CLASSIFIER_PATH,
]

print('DEVICE:', DEVICE)
for path in required_paths:
    print(path.exists(), path)

if DEVICE == 'cpu':
    raise RuntimeError('CUDA GPU를 사용할 수 없습니다.')
if not all(path.exists() for path in required_paths):
    raise FileNotFoundError('False로 표시된 입력 경로를 확인하세요.')


## 2. Validation GT 로드

9-class YOLO 라벨에서 GT class와 원본 이미지 좌표의 bbox를 읽습니다.
현재 데이터는 이미지당 객체 1개라는 조건을 검사합니다.


In [ ]:
IMAGE_EXTENSIONS = (
    '.jpg', '.jpeg', '.png', '.bmp', '.webp',
    '.JPG', '.JPEG', '.PNG', '.BMP', '.WEBP',
)


def find_image(stem):
    for extension in IMAGE_EXTENSIONS:
        candidate = VAL_IMAGE_DIR / f'{stem}{extension}'
        if candidate.is_file():
            return candidate
    return None


def yolo_box_to_xyxy(xc, yc, bw, bh, width, height):
    return [
        (xc - bw / 2) * width,
        (yc - bh / 2) * height,
        (xc + bw / 2) * width,
        (yc + bh / 2) * height,
    ]


records = []
gt_errors = []
label_paths = sorted(VAL_LABEL_DIR.glob('*.txt'))

for label_path in tqdm(label_paths, desc='GT load'):
    image_path = find_image(label_path.stem)
    if image_path is None:
        gt_errors.append(f'image missing: {label_path.name}')
        continue

    lines = [
        line.strip()
        for line in label_path.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
    if len(lines) != 1:
        gt_errors.append(f'object count {len(lines)}: {label_path.name}')
        continue

    parts = lines[0].split()
    if len(parts) != 5:
        gt_errors.append(f'label format: {label_path.name}')
        continue

    final_class = int(float(parts[0]))
    xc, yc, bw, bh = map(float, parts[1:])

    with Image.open(image_path) as opened:
        image = ImageOps.exif_transpose(opened)
        width, height = image.size

    records.append({
        'stem': label_path.stem,
        'image_path': str(image_path),
        'gt_final': final_class,
        'gt_material': final_class // 3,
        'gt_dirty': final_class % 3,
        'gt_bbox': yolo_box_to_xyxy(xc, yc, bw, bh, width, height),
        'image_width': width,
        'image_height': height,
    })

print('validation samples:', len(records))
print('GT errors:', len(gt_errors))
if gt_errors:
    print(gt_errors[:10])
    raise RuntimeError('GT 검사에 실패했습니다.')
if len(records) != 3158:
    raise RuntimeError(f'예상 validation 3158장과 다릅니다: {len(records)}')

gt_distribution = pd.Series(
    [FINAL_NAMES[item['gt_final']] for item in records]
).value_counts().reindex(FINAL_NAMES)
display(gt_distribution.to_frame('count'))


## 3. 모델 로드 및 클래스 순서 검사

ResNet18 출력은 학습 때 고정한 `clean, outer, inner` 순서를 사용합니다.


In [ ]:
print('Loading 1-stage YOLO...')
yolo_1stage = YOLO(str(YOLO_1STAGE_PATH))

print('Loading 2-stage material YOLO...')
yolo_2stage = YOLO(str(YOLO_2STAGE_PATH))

print('1-stage names:', yolo_1stage.names)
print('2-stage names:', yolo_2stage.names)

one_stage_names = [yolo_1stage.names[index] for index in range(9)]
two_stage_names = [yolo_2stage.names[index] for index in range(3)]

if one_stage_names != FINAL_NAMES:
    raise ValueError(f'1-stage 클래스 순서 불일치: {one_stage_names}')
if two_stage_names != MATERIAL_NAMES:
    raise ValueError(f'2-stage 클래스 순서 불일치: {two_stage_names}')

classifier = models.resnet18(weights=None)
classifier.fc = nn.Linear(classifier.fc.in_features, 3)
classifier_state = torch.load(
    CLASSIFIER_PATH,
    map_location=DEVICE,
    weights_only=True,
)
classifier.load_state_dict(classifier_state, strict=True)
classifier.to(DEVICE)
classifier.eval()

classifier_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

print('Classifier class order:', DIRT_NAMES)
print('모델 로드 완료')


## 4. 공통 함수

- IoU: 예측 bbox와 GT bbox의 겹침 정도
- 2-stage crop: YOLO 예측 bbox에 학습과 동일한 padding 0.05 적용


In [ ]:
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)

    intersection_x1 = max(ax1, bx1)
    intersection_y1 = max(ay1, by1)
    intersection_x2 = min(ax2, bx2)
    intersection_y2 = min(ay2, by2)

    intersection_width = max(0.0, intersection_x2 - intersection_x1)
    intersection_height = max(0.0, intersection_y2 - intersection_y1)
    intersection = intersection_width * intersection_height

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def crop_with_padding(image, xyxy, padding):
    width, height = image.size
    x1, y1, x2, y2 = map(float, xyxy)
    box_width = x2 - x1
    box_height = y2 - y1

    x1 = max(0, math.floor(x1 - box_width * padding))
    y1 = max(0, math.floor(y1 - box_height * padding))
    x2 = min(width, math.ceil(x2 + box_width * padding))
    y2 = min(height, math.ceil(y2 + box_height * padding))

    if x2 <= x1 or y2 <= y1:
        return None
    return image.crop((x1, y1, x2, y2))


def save_json(path, data):
    path.write_text(
        json.dumps(data, ensure_ascii=False),
        encoding='utf-8',
    )


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


## 5. 1-stage 원시 예측 생성

낮은 confidence 0.01에서 한 번 추론해 저장한 뒤, 여러 threshold 평가는 재추론 없이 수행합니다.


In [ ]:
CACHE_1STAGE = OUTPUT_DIR / 'raw_predictions_1stage.json'
PARTIAL_1STAGE = OUTPUT_DIR / 'raw_predictions_1stage.partial.json'
RECORD_BY_STEM = {item['stem']: item for item in records}

if REUSE_PREDICTION_CACHE and CACHE_1STAGE.is_file():
    predictions_1stage = load_json(CACHE_1STAGE)
    timing_1stage = load_json(OUTPUT_DIR / 'timing_1stage.json')
    print('1-stage cache 재사용:', CACHE_1STAGE)
else:
    if PARTIAL_1STAGE.is_file():
        predictions_1stage = load_json(PARTIAL_1STAGE)
        print('1-stage 부분 저장에서 재개:', len(predictions_1stage))
    else:
        predictions_1stage = []
    completed_stems = {item['stem'] for item in predictions_1stage}
    speed_sums = {'preprocess': 0.0, 'inference': 0.0, 'postprocess': 0.0}
    wall_start = time.perf_counter()
    remaining_records = [
        record for record in records
        if record['stem'] not in completed_stems
    ]

    for processed, record in enumerate(tqdm(
        remaining_records,
        desc='1-stage predict (1 image at a time)',
    ), start=1):
        # 고해상도 이미지를 한 장씩만 열어 RAM 사용량을 일정하게 유지한다.
        result = yolo_1stage.predict(
            source=str(record['image_path']),
            imgsz=640,
            conf=RAW_PREDICTION_CONF,
            iou=NMS_IOU,
            max_det=MAX_DETECTIONS,
            stream=False,
            verbose=False,
            device=0,
        )[0]
        image_predictions = []
        if result.boxes is not None:
            for box_index in range(len(result.boxes)):
                class_id = int(result.boxes.cls[box_index].item())
                confidence = float(result.boxes.conf[box_index].item())
                bbox = result.boxes.xyxy[box_index].cpu().tolist()
                image_predictions.append({
                    'bbox': bbox,
                    'detector_confidence': confidence,
                    'final_confidence': confidence,
                    'final_class': class_id,
                    'material_class': class_id // 3,
                    'dirty_class': class_id % 3,
                })

        predictions_1stage.append({
            'stem': record['stem'],
            'predictions': image_predictions,
        })
        for key in speed_sums:
            speed_sums[key] += float(result.speed.get(key, 0.0))
        if processed % 100 == 0:
            save_json(PARTIAL_1STAGE, predictions_1stage)

    prediction_by_stem = {item['stem']: item for item in predictions_1stage}
    predictions_1stage = [prediction_by_stem[record['stem']] for record in records]

    timing_1stage = {
        'wall_seconds': time.perf_counter() - wall_start,
        'images': len(records),
        'wall_ms_per_image': (time.perf_counter() - wall_start) * 1000 / len(records),
        'yolo_speed_ms_per_image': {
            key: value / len(records) for key, value in speed_sums.items()
        },
        'raw_confidence': RAW_PREDICTION_CONF,
    }
    save_json(CACHE_1STAGE, predictions_1stage)
    save_json(OUTPUT_DIR / 'timing_1stage.json', timing_1stage)
    PARTIAL_1STAGE.unlink(missing_ok=True)

print('1-stage prediction records:', len(predictions_1stage))
print(json.dumps(timing_1stage, ensure_ascii=False, indent=2))


## 6. 2-stage 원시 예측 생성

각 material YOLO 예측 bbox를 padding 0.05로 crop하고 ResNet18이 `clean/outer/inner`를 분류합니다.
최종 class ID는 `material_id × 3 + dirty_id`입니다.


In [ ]:
CACHE_2STAGE = OUTPUT_DIR / 'raw_predictions_2stage.json'
PARTIAL_2STAGE = OUTPUT_DIR / 'raw_predictions_2stage.partial.json'

if REUSE_PREDICTION_CACHE and CACHE_2STAGE.is_file():
    predictions_2stage = load_json(CACHE_2STAGE)
    timing_2stage = load_json(OUTPUT_DIR / 'timing_2stage.json')
    print('2-stage cache 재사용:', CACHE_2STAGE)
else:
    if PARTIAL_2STAGE.is_file():
        predictions_2stage = load_json(PARTIAL_2STAGE)
        print('2-stage 부분 저장에서 재개:', len(predictions_2stage))
    else:
        predictions_2stage = []
    completed_stems = {item['stem'] for item in predictions_2stage}
    speed_sums = {'preprocess': 0.0, 'inference': 0.0, 'postprocess': 0.0}
    classifier_seconds = 0.0
    classified_crops = 0
    crop_failures = 0
    wall_start = time.perf_counter()

    remaining_records = [
        record for record in records
        if record['stem'] not in completed_stems
    ]

    for processed, record in enumerate(tqdm(
        remaining_records,
        desc='2-stage predict (1 image at a time)',
    ), start=1):
        result = yolo_2stage.predict(
            source=str(record['image_path']),
            imgsz=640,
            conf=RAW_PREDICTION_CONF,
            iou=NMS_IOU,
            max_det=MAX_DETECTIONS,
            stream=False,
            verbose=False,
            device=0,
        )[0]
        image_predictions = []
        valid_items = []
        crop_tensors = []

        # result.orig_img는 YOLO bbox 좌표와 동일한 방향/크기의 BGR 이미지
        rgb_array = result.orig_img[:, :, ::-1].copy()
        source_image = Image.fromarray(rgb_array).convert('RGB')

        if result.boxes is not None:
            for box_index in range(len(result.boxes)):
                material_id = int(result.boxes.cls[box_index].item())
                detector_confidence = float(result.boxes.conf[box_index].item())
                bbox = result.boxes.xyxy[box_index].cpu().tolist()
                crop = crop_with_padding(
                    source_image,
                    bbox,
                    TWO_STAGE_CROP_PADDING,
                )
                if crop is None:
                    crop_failures += 1
                    continue
                valid_items.append((bbox, material_id, detector_confidence))
                crop_tensors.append(classifier_transform(crop))

        if crop_tensors:
            batch = torch.stack(crop_tensors).to(DEVICE)
            classifier_start = time.perf_counter()
            with torch.inference_mode():
                probabilities = torch.softmax(classifier(batch), dim=1)
            torch.cuda.synchronize()
            classifier_seconds += time.perf_counter() - classifier_start
            dirty_confidences, dirty_classes = probabilities.max(dim=1)

            for item_index, (bbox, material_id, detector_confidence) in enumerate(valid_items):
                dirty_id = int(dirty_classes[item_index].item())
                dirty_confidence = float(dirty_confidences[item_index].item())
                final_class = material_id * 3 + dirty_id
                image_predictions.append({
                    'bbox': bbox,
                    'detector_confidence': detector_confidence,
                    'classifier_confidence': dirty_confidence,
                    'final_confidence': detector_confidence * dirty_confidence,
                    'final_class': final_class,
                    'material_class': material_id,
                    'dirty_class': dirty_id,
                })
            classified_crops += len(crop_tensors)

        predictions_2stage.append({
            'stem': record['stem'],
            'predictions': image_predictions,
        })
        for key in speed_sums:
            speed_sums[key] += float(result.speed.get(key, 0.0))
        if processed % 100 == 0:
            save_json(PARTIAL_2STAGE, predictions_2stage)

    prediction_by_stem = {item['stem']: item for item in predictions_2stage}
    predictions_2stage = [prediction_by_stem[record['stem']] for record in records]

    total_wall = time.perf_counter() - wall_start
    timing_2stage = {
        'wall_seconds': total_wall,
        'images': len(records),
        'wall_ms_per_image': total_wall * 1000 / len(records),
        'yolo_speed_ms_per_image': {
            key: value / len(records) for key, value in speed_sums.items()
        },
        'classifier_seconds': classifier_seconds,
        'classifier_ms_per_image': classifier_seconds * 1000 / len(records),
        'classified_crops': classified_crops,
        'crop_failures': crop_failures,
        'raw_confidence': RAW_PREDICTION_CONF,
        'crop_padding': TWO_STAGE_CROP_PADDING,
    }
    save_json(CACHE_2STAGE, predictions_2stage)
    save_json(OUTPUT_DIR / 'timing_2stage.json', timing_2stage)
    PARTIAL_2STAGE.unlink(missing_ok=True)

print('2-stage prediction records:', len(predictions_2stage))
print(json.dumps(timing_2stage, ensure_ascii=False, indent=2))


## 7. IoU 매칭 기반 평가 함수

한 이미지에서 threshold 이상인 예측 중 GT와 IoU 0.5 이상인 예측을 찾고,
그중 detector confidence가 가장 높은 예측을 대표 예측으로 사용합니다.
위치를 못 찾으면 `no_localization`으로 기록하여 전체 성능에서 제외하지 않습니다.


In [ ]:
NO_LOCALIZATION_ID = 9


def evaluate_predictions(raw_predictions, detector_threshold, pipeline_name):
    y_true = []
    y_pred = []
    detail_rows = []
    localized_count = 0
    exact_correct = 0
    material_correct = 0
    dirty_correct = 0
    total_predictions = 0
    false_localization_predictions = 0
    matched_ious = []

    for record, prediction_record in zip(records, raw_predictions):
        candidates = [
            prediction
            for prediction in prediction_record['predictions']
            if prediction['detector_confidence'] >= detector_threshold
        ]
        candidates.sort(
            key=lambda item: item['detector_confidence'],
            reverse=True,
        )
        total_predictions += len(candidates)

        localized_candidates = []
        for prediction in candidates:
            iou = box_iou(record['gt_bbox'], prediction['bbox'])
            if iou >= IOU_THRESHOLD:
                localized_candidates.append((prediction, iou))
            else:
                false_localization_predictions += 1

        if localized_candidates:
            selected, selected_iou = localized_candidates[0]
            predicted_final = int(selected['final_class'])
            predicted_material = int(selected['material_class'])
            predicted_dirty = int(selected['dirty_class'])
            localized_count += 1
            matched_ious.append(selected_iou)
            is_material_correct = predicted_material == record['gt_material']
            is_dirty_correct = predicted_dirty == record['gt_dirty']
            is_exact_correct = predicted_final == record['gt_final']
            material_correct += int(is_material_correct)
            dirty_correct += int(is_dirty_correct)
            exact_correct += int(is_exact_correct)
            status = 'correct' if is_exact_correct else 'wrong_class'
            selected_confidence = selected['detector_confidence']
            selected_bbox = selected['bbox']
        else:
            predicted_final = NO_LOCALIZATION_ID
            predicted_material = -1
            predicted_dirty = -1
            selected_iou = 0.0
            selected_confidence = 0.0
            selected_bbox = None
            is_material_correct = False
            is_dirty_correct = False
            is_exact_correct = False
            status = 'no_localization'

        y_true.append(record['gt_final'])
        y_pred.append(predicted_final)
        detail_rows.append({
            'pipeline': pipeline_name,
            'stem': record['stem'],
            'image_path': record['image_path'],
            'gt_final': record['gt_final'],
            'gt_name': FINAL_NAMES[record['gt_final']],
            'pred_final': predicted_final,
            'pred_name': (
                FINAL_NAMES[predicted_final]
                if predicted_final != NO_LOCALIZATION_ID
                else 'no_localization'
            ),
            'gt_material': record['gt_material'],
            'pred_material': predicted_material,
            'gt_dirty': record['gt_dirty'],
            'pred_dirty': predicted_dirty,
            'status': status,
            'iou': selected_iou,
            'detector_confidence': selected_confidence,
            'num_predictions': len(candidates),
            'gt_bbox': json.dumps(record['gt_bbox']),
            'pred_bbox': json.dumps(selected_bbox),
        })

    total_images = len(records)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(range(9)),
        average='macro',
        zero_division=0,
    )

    per_class_recall = {}
    for class_id, class_name in enumerate(FINAL_NAMES):
        class_indices = [i for i, value in enumerate(y_true) if value == class_id]
        correct = sum(y_pred[i] == class_id for i in class_indices)
        per_class_recall[class_name] = correct / len(class_indices) if class_indices else 0.0

    summary = {
        'pipeline': pipeline_name,
        'detector_threshold': detector_threshold,
        'iou_threshold': IOU_THRESHOLD,
        'images': total_images,
        'localization_recall': localized_count / total_images,
        'no_localization_rate': 1 - localized_count / total_images,
        'exact_9class_accuracy_all_images': exact_correct / total_images,
        'exact_9class_macro_f1_all_images': macro_f1,
        'exact_accuracy_when_localized': exact_correct / localized_count if localized_count else 0.0,
        'material_accuracy_when_localized': material_correct / localized_count if localized_count else 0.0,
        'dirty_accuracy_when_localized': dirty_correct / localized_count if localized_count else 0.0,
        'mean_matched_iou': float(np.mean(matched_ious)) if matched_ious else 0.0,
        'predictions_per_image': total_predictions / total_images,
        'false_localization_predictions': false_localization_predictions,
        'per_class_exact_recall': per_class_recall,
    }
    return summary, pd.DataFrame(detail_rows), y_true, y_pred


## 8. Confidence threshold sweep

같은 숫자의 threshold가 두 모델에서 완전히 같은 확률 의미를 갖지는 않으므로,
공통 threshold 0.25 결과와 각 파이프라인의 validation 최적 threshold를 모두 남깁니다.


In [ ]:
sweep_rows = []

for threshold in THRESHOLDS:
    for pipeline_name, raw_predictions in (
        ('1-stage', predictions_1stage),
        ('2-stage', predictions_2stage),
    ):
        summary, _, _, _ = evaluate_predictions(
            raw_predictions,
            threshold,
            pipeline_name,
        )
        row = {
            key: value
            for key, value in summary.items()
            if key != 'per_class_exact_recall'
        }
        sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(OUTPUT_DIR / 'threshold_sweep.csv', index=False)

display(sweep_df[[
    'pipeline',
    'detector_threshold',
    'localization_recall',
    'exact_9class_accuracy_all_images',
    'exact_9class_macro_f1_all_images',
    'material_accuracy_when_localized',
    'dirty_accuracy_when_localized',
    'predictions_per_image',
]])


## 9. 최적 threshold 및 공통 0.25 결과 비교

최적 기준은 전체 이미지 기준 최종 9-class macro F1입니다.


In [ ]:
best_rows = (
    sweep_df.sort_values(
        ['pipeline', 'exact_9class_macro_f1_all_images'],
        ascending=[True, False],
    )
    .groupby('pipeline', as_index=False)
    .first()
)

common_rows = sweep_df[
    np.isclose(sweep_df['detector_threshold'], COMMON_REPORT_THRESHOLD)
].copy()

print('===== 파이프라인별 validation 최적 threshold =====')
display(best_rows)

print('===== 공통 detector threshold 0.25 =====')
display(common_rows)

best_rows.to_csv(OUTPUT_DIR / 'best_threshold_summary.csv', index=False)
common_rows.to_csv(OUTPUT_DIR / 'common_threshold_025_summary.csv', index=False)


## 10. 최적 threshold 상세 결과와 confusion matrix 저장

In [ ]:
final_summaries = {}
detail_frames = []

for pipeline_name, raw_predictions in (
    ('1-stage', predictions_1stage),
    ('2-stage', predictions_2stage),
):
    best_threshold = float(
        best_rows.loc[
            best_rows['pipeline'] == pipeline_name,
            'detector_threshold',
        ].iloc[0]
    )
    summary, details, y_true, y_pred = evaluate_predictions(
        raw_predictions,
        best_threshold,
        pipeline_name,
    )
    final_summaries[pipeline_name] = summary
    detail_frames.append(details)
    details.to_csv(
        OUTPUT_DIR / f'details_{pipeline_name.replace("-", "_")}.csv',
        index=False,
    )

    confusion = np.zeros((9, 10), dtype=int)
    for true_value, predicted_value in zip(y_true, y_pred):
        confusion[true_value, predicted_value] += 1

    confusion_df = pd.DataFrame(
        confusion,
        index=FINAL_NAMES,
        columns=FINAL_NAMES + ['no_localization'],
    )
    confusion_df.to_csv(
        OUTPUT_DIR / f'confusion_{pipeline_name.replace("-", "_")}.csv',
        encoding='utf-8',
    )

    normalized = confusion / confusion.sum(axis=1, keepdims=True)
    figure, axis = plt.subplots(figsize=(14, 10))
    image = axis.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    figure.colorbar(image, ax=axis)
    axis.set_xticks(range(10), FINAL_NAMES + ['no_localization'], rotation=45, ha='right')
    axis.set_yticks(range(9), FINAL_NAMES)
    axis.set_xlabel('Predicted')
    axis.set_ylabel('Ground Truth')
    axis.set_title(f'{pipeline_name} normalized end-to-end confusion matrix')

    for row in range(9):
        for column in range(10):
            value = normalized[row, column]
            if value >= 0.01:
                axis.text(
                    column,
                    row,
                    f'{value:.2f}',
                    ha='center',
                    va='center',
                    fontsize=7,
                    color='white' if value > 0.5 else 'black',
                )
    figure.tight_layout()
    figure.savefig(
        OUTPUT_DIR / f'confusion_{pipeline_name.replace("-", "_")}.png',
        dpi=170,
        bbox_inches='tight',
    )
    plt.show()

save_json(OUTPUT_DIR / 'final_summary.json', final_summaries)
print(json.dumps(final_summaries, ensure_ascii=False, indent=2))


## 11. 최종 비교표

처리시간은 낮은 raw confidence 0.01에서 cache를 생성한 측정값이므로 실제 앱 threshold에서는 달라질 수 있습니다.
성능 비교와 별도로 기록합니다.


In [ ]:
comparison_rows = []

for pipeline_name, timing in (
    ('1-stage', timing_1stage),
    ('2-stage', timing_2stage),
):
    summary = final_summaries[pipeline_name]
    comparison_rows.append({
        'pipeline': pipeline_name,
        'selected_threshold': summary['detector_threshold'],
        'localization_recall': summary['localization_recall'],
        'final_9class_accuracy': summary['exact_9class_accuracy_all_images'],
        'final_9class_macro_f1': summary['exact_9class_macro_f1_all_images'],
        'material_accuracy_localized': summary['material_accuracy_when_localized'],
        'dirty_accuracy_localized': summary['dirty_accuracy_when_localized'],
        'mean_iou': summary['mean_matched_iou'],
        'wall_ms_per_image_at_raw_conf_001': timing['wall_ms_per_image'],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(OUTPUT_DIR / 'end_to_end_comparison.csv', index=False)
display(comparison_df)


## 12. 무작위 실패 사례 시각화

최종 클래스 오류와 위치 탐지 실패를 포함해 각 파이프라인에서 최대 20장을 저장합니다.
초록색은 GT, 빨간색은 평가에 매칭된 예측 bbox입니다.


In [ ]:
for details in detail_frames:
    pipeline_name = details['pipeline'].iloc[0]
    failures = details[details['status'] != 'correct']
    sample = failures.sample(
        n=min(20, len(failures)),
        random_state=42,
    ).reset_index(drop=True)

    figure, axes = plt.subplots(4, 5, figsize=(20, 14))
    axes = axes.flatten()
    for axis in axes:
        axis.axis('off')

    for axis, row in zip(axes, sample.itertuples()):
        record = next(item for item in records if item['stem'] == row.stem)
        with Image.open(row.image_path) as opened:
            image = ImageOps.exif_transpose(opened).convert('RGB')

        draw = ImageDraw.Draw(image)
        draw.rectangle(record['gt_bbox'], outline='lime', width=10)
        predicted_bbox = json.loads(row.pred_bbox)
        if predicted_bbox is not None:
            draw.rectangle(predicted_bbox, outline='red', width=10)

        axis.imshow(image)
        axis.set_title(
            f'GT: {row.gt_name}\nPred: {row.pred_name}\n'
            f'status={row.status}, IoU={row.iou:.2f}',
            fontsize=8,
        )
        axis.axis('off')

    figure.suptitle(f'{pipeline_name} end-to-end failure samples', fontsize=16)
    figure.tight_layout(rect=(0, 0, 1, 0.97))
    output_path = OUTPUT_DIR / f'failure_samples_{pipeline_name.replace("-", "_")}.png'
    figure.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('저장:', output_path)


## 13. 결과 해석 원칙

- 2-stage detector의 높은 mAP만으로 최종 파이프라인 승리를 주장하지 않습니다.
- 최종 판단은 전체 이미지 기준 9-class accuracy와 macro F1을 중심으로 합니다.
- 위치 탐지 실패를 제외하고 계산한 조건부 정확도만 제시하면 실제 앱 성능이 과대평가됩니다.
- 각 모델의 validation 최적 threshold 결과와 공통 0.25 결과를 함께 제시합니다.
- 이 validation set으로 threshold까지 선택했으므로 독립 test 성능이라고 표현하지 않습니다.
